# 08. 전력 이상치 탐지 (P, P1, P2, P3)

## 이상치 기준
- 물리적 기준: 단방향 소비 계량기에서 P < 0W (소비 계량기에서 음수 전력은 불가)
- 통계적 기준: 계량기별 일별 평균값 기준 평균 ± 3σ 초과
- 발전/양방향 계량기는 물리적 기준 제외 (부호 규약상 음수 정상)

In [1]:
import sys
from pathlib import Path

ROOT = Path('/home/aceya/EMS')
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import numpy as np
from ems.db import load_env, connect

load_env()

START = '2018-01-01'
END   = '2024-01-01'

# 발전/양방향 계량기: 음수 전력이 정상인 계량기
BIDIRECTIONAL = {
    'H1.Z20', 'H1.ZE20',
    'V.Z84', 'V.ZE84',
    'V.Z81', 'V.Z82',
    'H1.Z310', 'H2.Z311', 'H3.Z312',
    'H1.Z29', 'H1.Z28',
    'H2.T.Z33',
}

ALL_METERS = [
    'H1.Z10', 'H1.Z11', 'H1.Z12', 'H1.Z13', 'H1.Z14', 'H1.Z15', 'H1.Z16',
    'H1.Z17', 'H1.Z18', 'H1.Z19', 'H1.Z20', 'H1.Z21', 'H1.Z22', 'H1.Z23',
    'H1.Z24', 'H1.Z25', 'H1.Z26', 'H1.Z27', 'H1.Z28', 'H1.Z29', 'H1.Z310',
    'H1.ZE20', 'H2.T.Z30', 'H2.T.Z31', 'H2.T.Z32', 'H2.T.Z33', 'H2.T.Z34',
    'H2.Z311', 'H2.Z35', 'H2.Z64', 'H2.Z65', 'H2.Z66', 'H2.Z67', 'H2.Z68',
    'H2.Z69', 'H2.Z70', 'H2.ZE64', 'H2.ZE65', 'H2.ZE66', 'H2.ZE67', 'H2.ZE74',
    'H3.Z312', 'H3.Z40', 'H3.Z41', 'H3.Z42', 'H4.Z50', 'H4.Z51', 'H4.ZE50',
    'H4.ZE51', 'V.Z81', 'V.Z82', 'V.Z84', 'V.ZE84'
]

save_dir = ROOT / 'outputs/tables/anomaly'
save_dir.mkdir(parents=True, exist_ok=True)
print('설정 완료')

설정 완료


In [2]:
def fetch_daily(meter_urn, measurement):
    sql = """
        SELECT
            DATE(ts AT TIME ZONE 'Europe/Berlin') AS day,
            MIN(value) AS min_val,
            MAX(value) AS max_val,
            AVG(value) AS avg_val,
            COUNT(*) AS cnt
        FROM ems.cr_measurement_1h
        WHERE meter_urn = %s
          AND measurement = %s
          AND ts >= %s
          AND ts <  %s
        GROUP BY 1
        ORDER BY 1
    """
    with connect() as conn:
        df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))
    df['day'] = pd.to_datetime(df['day'])
    return df


def detect_anomaly(meter, measurement):
    df = fetch_daily(meter, measurement)
    if df.empty:
        return None

    results = []

    # 1. 물리적 기준: 단방향 소비 계량기에서 P < 0
    if meter not in BIDIRECTIONAL:
        physical = df[df['min_val'] < 0].copy()
        physical['anomaly_type'] = '물리적이상(음수전력)'
        physical['criterion'] = 'min_val < 0W'
        if len(physical) > 0:
            results.append(physical)

    # 2. 통계적 기준: 평균 ± 3σ
    mean_val = df['avg_val'].mean()
    std_val  = df['avg_val'].std()
    upper = mean_val + 3 * std_val
    lower = mean_val - 3 * std_val
    stat = df[(df['avg_val'] > upper) | (df['avg_val'] < lower)].copy()
    stat['anomaly_type'] = '통계적이상(3sigma)'
    stat['criterion'] = f'mean={mean_val:.2f}, sigma={std_val:.2f}, lower={lower:.2f}, upper={upper:.2f}'
    if len(stat) > 0:
        results.append(stat)

    if not results:
        return None

    result = pd.concat(results).drop_duplicates('day').sort_values('day')
    result['meter'] = meter
    result['measurement'] = measurement
    return result[['meter', 'measurement', 'day', 'min_val', 'max_val', 'avg_val', 'anomaly_type', 'criterion']]

In [3]:
all_results = []

for meter in ALL_METERS:
    for meas in ['P', 'P1', 'P2', 'P3']:
        result = detect_anomaly(meter, meas)
        if result is not None and len(result) > 0:
            print(f'{meter} {meas}: {len(result)}건')
            all_results.append(result)

if all_results:
    final = pd.concat(all_results, ignore_index=True)
    final.to_csv(save_dir / 'anomaly_power.csv', index=False)
    print(f'\n총 {len(final)}건 저장 완료')
else:
    print('이상치 없음')

/tmp/ipykernel_103634/1728458483.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z10 P: 32건


/tmp/ipykernel_103634/1728458483.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z10 P1: 27건


/tmp/ipykernel_103634/1728458483.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


...

H4.ZE51 P2: 2건


/tmp/ipykernel_103634/1728458483.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H4.ZE51 P3: 2건


/tmp/ipykernel_103634/1728458483.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))
/tmp/ipykernel_103634/1728458483.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))
/tmp/ipykernel_103634/1728458483.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))
/tmp/ipykernel_103634/1728458483.py:18: UserWarning:

V.ZE84 P3: 16건

총 18845건 저장 완료


In [4]:
if all_results:
    summary = final.groupby(['meter', 'measurement', 'anomaly_type']).agg(
        건수=('day', 'count'),
        시작일=('day', 'min'),
        종료일=('day', 'max'),
        min_val=('min_val', 'min'),
        max_val=('max_val', 'max'),
        criterion=('criterion', 'first')
    ).reset_index()
    summary.to_csv(save_dir / 'anomaly_power_summary.csv', index=False)
    print(summary.to_string())

        meter measurement   anomaly_type    건수        시작일        종료일       min_val        max_val                                                       criterion
0      H1.Z10           P    물리적이상(음수전력)     5 2022-08-26 2022-09-02 -5.664686e+03   21949.481167                                                    min_val < 0W
1      H1.Z10           P  통계적이상(3sigma)    27 2018-08-29 2023-01-12  4.462367e+01   25976.326833         mean=4468.39, sigma=1469.26, lower=60.60, upper=8876.18
2      H1.Z10          P1    물리적이상(음수전력)     5 2022-08-26 2022-09-02 -2.416816e+03    6883.698500                                                    min_val < 0W
3      H1.Z10          P1  통계적이상(3sigma)    22 2018-08-29 2022-07-07  9.955288e+02    9035.884049        mean=1269.09, sigma=496.97, lower=-221.82, upper=2759.99
4      H1.Z10          P2    물리적이상(음수전력)    45 2018-11-09 2023-06-26 -1.683820e+03    7612.195500                                                    min_val < 0W
5      H1.Z10          P2  통